# 01 — Fixed-Penalty K4: Frozen Matched Multi-Seed Replication

**Frozen protocol**

- arms: `K0_UNCONSTRAINED` and `K4_VERY_STRONG` only;
- five matched training seeds (minimum admissible completed set: three);
- checkpoint 10 is the immutable primary endpoint;
- checkpoint 20 is sensitivity-only and can never replace checkpoint 10;
- the penalty, endpoints, budgets, activity floor and validation windows are read from frozen artifacts;
- no grid expansion, checkpoint selection, test evaluation or OOS access;
- inference is hierarchical paired bootstrap: seeds first, matched windows second;
- each episode, rollout and checkpoint is saved atomically for safe resume.

In [ ]:
from google.colab import drive
from pathlib import Path
import os, sys

MOUNT = Path("/content/drive")
if not (MOUNT / "MyDrive").exists():
    drive.mount(str(MOUNT))

PROJECT_ROOT = Path(
    os.environ.get("RIMM_PROJECT_ROOT", MOUNT / "MyDrive" / "RIMM")
)
if not PROJECT_ROOT.is_dir():
    raise FileNotFoundError(PROJECT_ROOT)

os.chdir(PROJECT_ROOT)

SRC = PROJECT_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

print("PROJECT_ROOT:", PROJECT_ROOT)


In [ ]:
import copy
import gc
import json
import random
import subprocess
import time
import hashlib
import warnings
from dataclasses import dataclass, field

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import yaml
import pyarrow.parquet as pq
from IPython.display import display

warnings.filterwarnings("ignore", category=FutureWarning)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE)
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))


## Frozen configuration and execution authorization

In [ ]:
SEED = 42
SYMBOL = "ETHUSDT"
RUN_PROFILE = "frozen_k4_ep10_multiseed"
SPEC_ID = "v08_frozen_k4_ep10_multiseed_20260925"

MASTER_SEEDS = (10801, 10802, 10803, 10804, 10805)
LABELS = ("K0_UNCONSTRAINED", "K4_VERY_STRONG")
PRIMARY_CHECKPOINT = 10
SENSITIVITY_CHECKPOINT = 20

V08_CFG = dict(
    training_seeds=MASTER_SEEDS,
    minimum_complete_seeds=3,
    target_episodes=20,
    snapshot_episodes=(10, 20),
    bootstrap_samples=10_000,
    activity_floor_fraction=0.50,
)

TRAIN_EPISODE_MINUTES = 30.0
EVAL_EPISODE_MINUTES = 30.0
DECISION_INTERVAL_MS = 500
TRAIN_CFG = dict(
    replay_capacity=150_000, batch_size=256, gamma=0.999, tau=0.005,
    policy_delay=2, target_noise=0.2, noise_clip=0.5,
    exploration_noise=0.1, actor_lr=3e-4, critic_lr=3e-4,
    encoder_lr=3e-4, cost_critic_lr=3e-4, update_every=2,
    pretrain_steps=20_000, branch_steps=30_000,
    random_warmup_steps=5_000, log_every=2_000,
)
IMM_CFG = dict(
    expert_critic_batch_size=128, expert_actor_batch_size=256,
    bc_weight_0=1.0, imitation_decay_fraction=0.40,
)
RISK_CFG = dict(
    gamma_cost=0.999, reference_band_N=3.0,
    loose_fraction=0.80, strict_fraction=0.50,
)
EVAL_CFG = dict(bootstrap_samples=V08_CFG["bootstrap_samples"])
EXPECTED_UPDATES = 22_500

torch.manual_seed(SEED); np.random.seed(SEED); random.seed(SEED)
print(json.dumps({
    "spec_id": SPEC_ID, "seeds": MASTER_SEEDS, "labels": LABELS,
    "primary_checkpoint": PRIMARY_CHECKPOINT,
    "sensitivity_checkpoint": SENSITIVITY_CHECKPOINT,
    "test_accessed": False, "checkpoint_reselection_allowed": False,
}, indent=2))

{
  "spec_id": "v08_frozen_k4_ep10_multiseed_20260925",
  "seeds": [
    10801,
    10802,
    10803,
    10804,
    10805
  ],
  "labels": [
    "K0_UNCONSTRAINED",
    "K4_VERY_STRONG"
  ],
  "primary_checkpoint": 10,
  "sensitivity_checkpoint": 20,
  "test_accessed": false,
  "checkpoint_reselection_allowed": false
}


## Risk utilities and regression tests

The public repository already contains the frozen implementation in `src/mmrl/imm/risk_budget.py`; the notebook verifies it rather than rewriting source files at runtime.


In [ ]:
RISK_MODULE = PROJECT_ROOT / "src" / "mmrl" / "imm" / "risk_budget.py"
if not RISK_MODULE.is_file():
    raise FileNotFoundError(RISK_MODULE)
print("risk module: available")


risk module: available


In [ ]:
TEST_PATH = PROJECT_ROOT / "tests" / "test_imm_risk_budget.py"
if not TEST_PATH.is_file():
    raise FileNotFoundError(TEST_PATH)

subprocess.run(
    [sys.executable, "-m", "pytest", "-q", str(TEST_PATH)],
    check=True,
)
print("risk regression tests: PASS")


risk regression tests: PASS


## Frozen IMM components, data loader and networks

In [ ]:
from mmrl.imm.environment import IMMMarketData, IMMMarketMakingEnv, IMMRewardConfig
from mmrl.imm.tcsa import IMMTCSAEncoder, IMMStateEncoder, TD3Actor, TD3TwinCritic
from mmrl.imm.td3 import clipped_target_action, polyak_update, reconstruct_market_batch
from mmrl.imm.reward import NotionalBpsReward
from mmrl.imm.risk_budget import InventoryExposureCost, ProjectedLagrange, RiskIndexedReplayBuffer
from mmrl.simulator import ReplaySimulator

RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
SPLIT_DIR = PROJECT_ROOT / "artifacts" / "splits"
SIGNAL_DIR = PROJECT_ROOT / "artifacts" / "imm_predictive_signals"
EXPERT_DIR = PROJECT_ROOT / "artifacts" / "imm_expert_bc"

ARTIFACT_DIR = PROJECT_ROOT / "artifacts" / "risk_budgeted_imm_v01"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

with open(PROJECT_ROOT / "configs" / "data.yaml", "r", encoding="utf-8") as f:
    DATA_CFG = yaml.safe_load(f)

with open(PROJECT_ROOT / "configs" / "simulator.yaml", "r", encoding="utf-8") as f:
    SIM_CFG = yaml.safe_load(f)

SYMBOLS = list(DATA_CFG["symbols"])
OFFSETS_MS = DATA_CFG["trade_alignment_offset_ms"]
MAX_GAP_MS = int(SIM_CFG["max_gap_ms"])

QUOTE_SIZE = {
    "BTCUSDT": 0.08220875,
    "ETHUSDT": 1.2455625,
    "BNBUSDT": 0.9417,
    "LTCUSDT": 3.8242,
}

# Frozen economic scaling; only the fixed risk penalty is removed.
ECONOMIC_REWARD_FN = NotionalBpsReward(
    beta=0.01,
    eta=0.0,
    inventory_threshold_N=3.0,
)

RISK_COST_FN = InventoryExposureCost(
    reference_band_N=RISK_CFG["reference_band_N"],
)

from mmrl.imm.grid import price_tick_to_grid_level


In [ ]:
def infer_tick_size(states):
    gaps = []
    for i in range(1, 5):
        gaps.append(
            states[f"ask_price_{i+1}"].to_numpy(float)
            - states[f"ask_price_{i}"].to_numpy(float)
        )
        gaps.append(
            states[f"bid_price_{i}"].to_numpy(float)
            - states[f"bid_price_{i+1}"].to_numpy(float)
        )

    x = np.concatenate(gaps)
    x = x[np.isfinite(x) & (x > 0)]
    values, counts = np.unique(np.round(x, 10), return_counts=True)
    return float(values[np.argmax(counts)])


def normalize_bool(series):
    if pd.api.types.is_bool_dtype(series):
        return series.astype(bool)
    if pd.api.types.is_numeric_dtype(series):
        return series.astype(np.int8).astype(bool)

    mapping = {
        "true": True, "1": True, "t": True, "yes": True,
        "false": False, "0": False, "f": False, "no": False,
    }
    out = series.astype(str).str.lower().str.strip().map(mapping)
    if out.isna().any():
        raise ValueError("unknown boolean value")
    return out.astype(bool)


def load_asset(symbol):
    state_path = PROCESSED_DIR / symbol / "states.parquet"
    split_path = SPLIT_DIR / f"{symbol.lower()}_state_splits.parquet"
    trade_path = RAW_DIR / symbol / "trades.parquet"
    signal_path = SIGNAL_DIR / symbol / "signals.parquet"
    demo_path = EXPERT_DIR / symbol / "expert_demonstrations.parquet"

    state_cols = ["local_timestamp", "mid_price"]
    for side in ["bid", "ask"]:
        for i in range(1, 6):
            state_cols += [f"{side}_price_{i}", f"{side}_qty_{i}"]

    states = (
        pd.read_parquet(state_path, columns=state_cols)
        .sort_values("local_timestamp", kind="stable")
        .reset_index(drop=True)
    )
    split = (
        pd.read_parquet(split_path)
        .sort_values("local_timestamp", kind="stable")
        .reset_index(drop=True)
    )
    signals_df = (
        pd.read_parquet(signal_path)
        .sort_values("local_timestamp", kind="stable")
        .reset_index(drop=True)
    )

    if not np.array_equal(
        pd.to_numeric(states["local_timestamp"]).to_numpy(np.int64),
        pd.to_numeric(signals_df["local_timestamp"]).to_numpy(np.int64),
    ):
        raise ValueError(f"{symbol}: signal timestamp mismatch")

    tick_size = infer_tick_size(states)
    schema = pq.ParquetFile(trade_path).schema.names

    if "timestamp_us" in schema:
        ts_col, multiplier = "timestamp_us", 1
    elif "timestamp_ms" in schema:
        ts_col, multiplier = "timestamp_ms", 1000
    else:
        raise ValueError(f"{symbol}: missing trade timestamp")

    trades = pd.read_parquet(
        trade_path,
        columns=[ts_col, "price", "qty", "is_buyer_maker"],
    ).copy()
    trades["effective_timestamp_us"] = (
        pd.to_numeric(trades[ts_col]).astype(np.int64)
        * int(multiplier)
        + int(OFFSETS_MS[symbol]) * 1000
    )
    trades["price"] = pd.to_numeric(trades["price"]).astype(float)
    trades["qty"] = pd.to_numeric(trades["qty"]).astype(float)
    trades["is_buyer_maker"] = normalize_bool(trades["is_buyer_maker"])
    trades = (
        trades[["effective_timestamp_us", "price", "qty", "is_buyer_maker"]]
        .sort_values("effective_timestamp_us", kind="stable")
        .reset_index(drop=True)
    )

    data = IMMMarketData(
        symbol=symbol,
        states=states,
        split=split,
        trades=trades,
        quote_size=QUOTE_SIZE[symbol],
        tick_size=tick_size,
        lookback=64,
    )

    signals = signals_df[
        ["signal_10s", "signal_60s", "signal_120s", "signal_300s"]
    ].to_numpy(np.float32)
    ready = signals_df["signal_ready"].to_numpy(bool)

    demos = (
        pd.read_parquet(demo_path)
        .sort_values("timestamp_ms", kind="stable")
        .reset_index(drop=True)
    )
    return data, signals, ready, demos


In [ ]:
@dataclass
class RiskBundle:
    se: torch.nn.Module
    actor: torch.nn.Module
    reward_critics: torch.nn.Module
    cost_critics: torch.nn.Module
    tse: torch.nn.Module
    tactor: torch.nn.Module
    treward_critics: torch.nn.Module
    tcost_critics: torch.nn.Module
    se_opt: torch.optim.Optimizer
    actor_opt: torch.optim.Optimizer
    reward_critics_opt: torch.optim.Optimizer
    cost_critics_opt: torch.optim.Optimizer


def make_bundle(cfg=TRAIN_CFG):
    market_encoder = IMMTCSAEncoder(
        feature_dim=20,
        lookback=64,
        latent_dim=64,
        tcn_hidden_channels=64,
        tcn_kernel_size=3,
        tcn_dilations=(1, 2, 4),
        tcn_dropout=0.0,
    )
    se = IMMStateEncoder(market_encoder, signal_dim=4, private_dim=21)
    actor = TD3Actor(89, action_dim=4, hidden_dims=(256, 256))
    reward_critics = TD3TwinCritic(89, action_dim=4, hidden_dims=(256, 256))
    cost_critics = TD3TwinCritic(89, action_dim=4, hidden_dims=(256, 256))

    tse = copy.deepcopy(se)
    tactor = copy.deepcopy(actor)
    treward_critics = copy.deepcopy(reward_critics)
    tcost_critics = copy.deepcopy(cost_critics)

    se = se.to(DEVICE)
    actor = actor.to(DEVICE)
    reward_critics = reward_critics.to(DEVICE)
    cost_critics = cost_critics.to(DEVICE)
    tse = tse.to(DEVICE)
    tactor = tactor.to(DEVICE)
    treward_critics = treward_critics.to(DEVICE)
    tcost_critics = tcost_critics.to(DEVICE)

    for module in [tse, tactor, treward_critics, tcost_critics]:
        module.eval()
        for p in module.parameters():
            p.requires_grad_(False)

    return RiskBundle(
        se=se,
        actor=actor,
        reward_critics=reward_critics,
        cost_critics=cost_critics,
        tse=tse,
        tactor=tactor,
        treward_critics=treward_critics,
        tcost_critics=tcost_critics,
        se_opt=torch.optim.Adam(se.parameters(), lr=cfg["encoder_lr"]),
        actor_opt=torch.optim.Adam(actor.parameters(), lr=cfg["actor_lr"]),
        reward_critics_opt=torch.optim.Adam(
            reward_critics.parameters(), lr=cfg["critic_lr"]
        ),
        cost_critics_opt=torch.optim.Adam(
            cost_critics.parameters(), lr=cfg["cost_critic_lr"]
        ),
    )


def make_simulator():
    return ReplaySimulator(
        max_gap_ms=MAX_GAP_MS,
        maker_fee_rate=0.0,
        allow_marketable_orders=bool(SIM_CFG["allow_marketable_orders"]),
        allow_inside_spread=bool(SIM_CFG["allow_inside_spread"]),
        allow_beyond_depth5=bool(SIM_CFG["allow_beyond_depth5"]),
        trade_through_fill=bool(SIM_CFG["trade_through_fill"]),
        allow_partial_fills=bool(SIM_CFG["allow_partial_fills"]),
    )


def make_env(data, signals, seed, episode_minutes=TRAIN_EPISODE_MINUTES):
    return ContinuityIMMMarketMakingEnv(
        data=data,
        simulator=make_simulator(),
        decision_interval_ms=DECISION_INTERVAL_MS,
        episode_duration_minutes=float(episode_minutes),
        reward_config=IMMRewardConfig(mode="pnl"),
        signal_matrix=signals,
        seed=seed,
    )

def available_split_name(data, preferred):
    names = {str(x) for x in np.unique(data.split_name)}
    aliases = {
        "train": ["train"],
        "validation": ["validation", "val", "valid"],
        "test": ["test"],
    }
    for name in aliases.get(preferred, [preferred]):
        if name in names:
            return name
    raise RuntimeError(
        f"No {preferred!r} split. Available split names: {sorted(names)}"
    )


def eligible_starts(env, ready, split_name):
    data = env.data
    candidate = np.flatnonzero(
        (data.split_name == split_name) & data.lookback_ready & ready
    ).astype(np.int64)

    n = len(data.timestamp_ms)
    episode_right = np.empty(n, dtype=np.int64)
    segment_break = (
        (data.episode_id[1:] != data.episode_id[:-1])
        | (data.split_name[1:] != data.split_name[:-1])
    )
    left_edges = np.flatnonzero(np.r_[True, segment_break])
    right_edges = np.r_[left_edges[1:] - 1, n - 1]
    for left, right in zip(left_edges, right_edges):
        episode_right[left:right + 1] = right

    available = (
        data.timestamp_ms[episode_right[candidate]]
        - data.timestamp_ms[candidate]
    )
    out = candidate[available >= env.episode_duration_ms]
    if len(out) == 0:
        raise RuntimeError(f"no full-duration starts for split={split_name}")
    return out


def fixed_start_subset(starts, n):
    starts = np.asarray(starts, dtype=np.int64)
    if len(starts) <= int(n):
        return starts.copy()
    positions = np.linspace(0, len(starts) - 1, num=int(n))
    return starts[np.unique(np.rint(positions).astype(np.int64))]


@torch.no_grad()
def get_action(bundle, obs, noise=0.0):
    market = torch.from_numpy(
        np.asarray(obs["market"], np.float32)[None]
    ).to(DEVICE)
    signals_t = torch.from_numpy(
        np.asarray(obs["signals"], np.float32)[None]
    ).to(DEVICE)
    private = torch.from_numpy(
        np.asarray(obs["private"], np.float32)[None]
    ).to(DEVICE)

    state = bundle.se(market, signals_t, private)
    action = bundle.actor(state)[0]
    if noise > 0:
        action = action + torch.randn_like(action) * noise
    return action.clamp(-1, 1).cpu().numpy().astype(np.float32)


def tensor_state(data, signals, idx, private):
    idx = np.asarray(idx, np.int64)
    market = reconstruct_market_batch(
        data.market_features,
        idx,
        lookback=64,
    )
    signal = signals[idx].astype(np.float32)
    return (
        torch.from_numpy(market).to(DEVICE),
        torch.from_numpy(signal).to(DEVICE),
        torch.from_numpy(np.asarray(private, np.float32)).to(DEVICE),
    )


def set_grad(module, value):
    for p in module.parameters():
        p.requires_grad_(bool(value))


def positive_cost_pair(critics, state, action):
    q1, q2 = critics(state, action)
    return F.softplus(q1), F.softplus(q2)


def positive_cost_q1(critics, state, action):
    return F.softplus(critics.q1(state, action))


## Continuity-preserving environment

In [ ]:
class ContinuityIMMMarketMakingEnv(IMMMarketMakingEnv):
    def _cancel_offgrid_orders_at_current_decision(self):
        if self.idx is None:
            raise RuntimeError("env not reset")

        ref2 = int(self.data.ref2[int(self.idx)])
        active_orders = list(self.simulator.order_manager.active_orders)
        offgrid_orders = [
            order
            for order in active_orders
            if price_tick_to_grid_level(
                ref2,
                int(order.price_tick),
                k=5,
            ) is None
        ]

        canceled_qty = 0.0
        for order in offgrid_orders:
            canceled_qty += float(
                self.simulator.order_manager.cancel_order(int(order.order_id))
            )

        obs, private_info = self._observation()
        return (
            obs.as_dict(),
            private_info,
            int(len(offgrid_orders)),
            float(canceled_qty),
        )

    def step(self, action):
        obs, reward, terminated, truncated, info = super().step(action)

        before_count = int(info["offgrid_live_child_count"])
        before_qty = float(info["offgrid_live_qty"])

        (
            repaired_obs,
            private_info,
            canceled_count,
            canceled_qty,
        ) = self._cancel_offgrid_orders_at_current_decision()

        if canceled_count != before_count:
            raise AssertionError(
                f"reflection/direct off-grid count mismatch: "
                f"{before_count} vs {canceled_count}"
            )

        info.update(private_info)
        info["offgrid_live_child_count_before_auto_cancel"] = before_count
        info["offgrid_live_qty_before_auto_cancel"] = before_qty
        info["offgrid_auto_canceled_count"] = canceled_count
        info["offgrid_auto_canceled_qty"] = canceled_qty
        info["offgrid_live_child_count_after_auto_cancel"] = int(
            private_info["offgrid_live_child_count"]
        )
        info["offgrid_live_qty_after_auto_cancel"] = float(
            private_info["offgrid_live_qty"]
        )
        info["canceled_qty"] = float(info["canceled_qty"]) + canceled_qty

        if info["offgrid_live_child_count_after_auto_cancel"] != 0:
            raise AssertionError("off-grid order survived auto-cancel")

        return repaired_obs, reward, terminated, truncated, info


## Expert replay and common helpers

In [ ]:
class RiskExpertSampler:
    def __init__(self, data, demos, reward_fn, cost_fn, seed=42):
        self.data = data
        self.rng = np.random.default_rng(seed)

        self.actor_df = demos[demos["bc_valid"]].reset_index(drop=True)
        self.critic_df = demos[demos["critic_valid"]].reset_index(drop=True)
        if len(self.actor_df) == 0 or len(self.critic_df) == 0:
            raise RuntimeError("empty expert subset")

        self.a_idx = self.actor_df["state_idx"].to_numpy(np.int64)
        self.a_private = self.actor_df[
            [f"private_{j}" for j in range(21)]
        ].to_numpy(np.float32)
        self.a_action = self.actor_df[
            [f"expert_action_{j}" for j in range(4)]
        ].to_numpy(np.float32)

        self.c_idx = self.critic_df["state_idx"].to_numpy(np.int64)
        self.c_next = self.critic_df["next_state_idx"].to_numpy(np.int64)
        self.c_private = self.critic_df[
            [f"private_{j}" for j in range(21)]
        ].to_numpy(np.float32)
        self.c_next_private = self.critic_df[
            [f"next_private_{j}" for j in range(21)]
        ].to_numpy(np.float32)
        self.c_action = self.critic_df[
            [f"expert_action_{j}" for j in range(4)]
        ].to_numpy(np.float32)

        rewards, costs = [], []
        for row in self.critic_df.itertuples(index=False):
            reward, _ = reward_fn.compute(
                pnl=float(row.pnl),
                filled_notional=float(row.filled_notional),
                inventory=float(row.inventory_after),
                quote_size=float(data.quote_size),
                mid_price=float(data.mid_price[int(row.next_state_idx)]),
            )
            cost, _ = cost_fn.compute(
                inventory=float(row.inventory_after),
                quote_size=float(data.quote_size),
            )
            rewards.append(reward)
            costs.append(cost)

        self.c_reward = np.asarray(rewards, dtype=np.float32)
        self.c_cost = np.asarray(costs, dtype=np.float32)
        self.c_done = (
            data.episode_id[self.c_idx] != data.episode_id[self.c_next]
        ).astype(np.float32)

    def sample_actor(self, n):
        i = self.rng.integers(0, len(self.actor_df), size=int(n))
        return self.a_idx[i], self.a_private[i], self.a_action[i]

    def sample_critic(self, n):
        i = self.rng.integers(0, len(self.critic_df), size=int(n))
        return (
            self.c_idx[i],
            self.c_next[i],
            self.c_private[i],
            self.c_next_private[i],
            self.c_action[i],
            self.c_reward[i],
            self.c_cost[i],
            self.c_done[i],
        )


In [ ]:
def bc_weight_at(global_gradient_step):
    decay_updates = max(
        1,
        round(IMM_CFG["imitation_decay_fraction"] * EXPECTED_UPDATES),
    )
    return IMM_CFG["bc_weight_0"] * max(
        0.0, 1.0 - global_gradient_step / decay_updates
    )


def reconstruct_batch(data, signals, batch):
    market, sig, private = tensor_state(
        data, signals, batch["state_idx"], batch["private"]
    )
    next_market, next_sig, next_private = tensor_state(
        data, signals, batch["next_state_idx"], batch["next_private"]
    )
    return dict(
        market=market,
        signals=sig,
        private=private,
        next_market=next_market,
        next_signals=next_sig,
        next_private=next_private,
        action=torch.from_numpy(batch["action"]).to(DEVICE),
        reward=torch.from_numpy(batch["reward"]).to(DEVICE).unsqueeze(-1),
        cost=torch.from_numpy(batch["cost"]).to(DEVICE).unsqueeze(-1),
        done=torch.from_numpy(batch["done"]).to(DEVICE).unsqueeze(-1),
    )


def torch_load_compat(path, map_location="cpu"):
    try:
        return torch.load(path, map_location=map_location, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=map_location)


def bootstrap_mean_ci(values, rng, n_boot):
    x = np.asarray(values, dtype=float)
    draws = rng.choice(
        x, size=(int(n_boot), len(x)), replace=True
    ).mean(axis=1)
    return float(np.quantile(draws, 0.025)), float(np.quantile(draws, 0.975))


def paired_bootstrap_arrays(a, b, seed):
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    if a.shape != b.shape:
        raise AssertionError(f"paired shape mismatch: {a.shape} vs {b.shape}")
    diff = a - b
    rng = np.random.default_rng(seed)
    draws = rng.choice(
        diff,
        size=(EVAL_CFG["bootstrap_samples"], len(diff)),
        replace=True,
    ).mean(axis=1)
    return dict(
        mean_difference=float(diff.mean()),
        ci_low=float(np.quantile(draws, 0.025)),
        ci_high=float(np.quantile(draws, 0.975)),
        paired_starts=int(len(diff)),
    )


def atomic_csv(frame, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + ".tmp")
    frame.to_csv(tmp, index=False)
    os.replace(tmp, path)


def atomic_parquet(frame, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + ".tmp")
    frame.to_parquet(tmp, index=False)
    os.replace(tmp, path)


def atomic_torch_save(payload, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + ".tmp")
    torch.save(payload, tmp)
    os.replace(tmp, path)


def module_digest(*modules):
    h = hashlib.sha256()
    for module in modules:
        for name, tensor in sorted(module.state_dict().items()):
            h.update(name.encode("utf-8"))
            h.update(tensor.detach().cpu().numpy().tobytes())
    return h.hexdigest()


## Load data, authorization and frozen experimental quantities

In [ ]:
data, signals, ready, demos = load_asset(SYMBOL)
TRAIN_SPLIT = available_split_name(data, "train")
VALIDATION_SPLIT = available_split_name(data, "validation")
if "test" in {str(TRAIN_SPLIT).lower(), str(VALIDATION_SPLIT).lower()}:
    raise RuntimeError("Test split access is forbidden in v0.8")

probe_env = make_env(data, signals, SEED, EVAL_EPISODE_MINUTES)
train_starts_all = eligible_starts(probe_env, ready, TRAIN_SPLIT)

V04_DIR = PROJECT_ROOT / "artifacts" / "risk_budgeted_imm_v04" / SYMBOL / "pilot"
V07_DIR = PROJECT_ROOT / "artifacts" / "risk_budgeted_imm_v07" / SYMBOL / "bounded_penalty_verification"
V07B_DIR = PROJECT_ROOT / "artifacts" / "risk_budgeted_imm_v07b" / SYMBOL / "q_adjudication"
V08_DIR = PROJECT_ROOT / "artifacts" / "risk_budgeted_imm_v08" / SYMBOL / RUN_PROFILE
V08_DIR.mkdir(parents=True, exist_ok=True)

freeze_path = V07B_DIR / "v08_freeze_spec_adjudicated.json"
required = [
    V04_DIR / "shared_pretrain_checkpoint.pt",
    V04_DIR / "budget_calibration.csv",
    V07_DIR / "validation_starts_timestamp_spaced.csv",
    freeze_path,
]
missing = [str(p) for p in required if not p.exists()]
if missing:
    raise FileNotFoundError("Missing frozen inputs:\n" + "\n".join(missing))

freeze = json.loads(freeze_path.read_text())
if not bool(freeze.get("valid_for_execution")) or not bool(freeze.get("approved_for_v08")):
    raise RuntimeError("v0.7b did not authorize v0.8 execution")
if freeze.get("selected_label") != "K4_VERY_STRONG":
    raise RuntimeError("Frozen candidate mismatch")
if int(freeze.get("primary_checkpoint")) != PRIMARY_CHECKPOINT:
    raise RuntimeError("Frozen primary checkpoint mismatch")
if int(freeze.get("sensitivity_checkpoint")) != SENSITIVITY_CHECKPOINT:
    raise RuntimeError("Frozen sensitivity checkpoint mismatch")
if not bool(freeze.get("no_new_kappa")) or not bool(freeze.get("no_checkpoint_selection_after_run")):
    raise RuntimeError("Frozen anti-retuning requirements are missing")
if bool(freeze.get("source_test_used")) or bool(freeze.get("audit_test_used")):
    raise RuntimeError("Source protocol reports test access")

SELECTED_KAPPA = float(freeze["selected_kappa"])
PENALTY_GRID = {"K0_UNCONSTRAINED": 0.0, "K4_VERY_STRONG": SELECTED_KAPPA}
GRID_LABELS = list(LABELS)

budget_df = pd.read_csv(V04_DIR / "budget_calibration.csv")
LOOSE_BUDGET = float(freeze["primary_budget"])
STRICT_BUDGET = float(freeze["secondary_strict_budget"])
if not np.isclose(LOOSE_BUDGET, float(budget_df.loc[0,"loose_budget"]), rtol=0, atol=1e-12):
    raise RuntimeError("Loose budget identity mismatch")
if not np.isclose(STRICT_BUDGET, float(budget_df.loc[0,"strict_budget"]), rtol=0, atol=1e-12):
    raise RuntimeError("Strict budget identity mismatch")

validation_start_df = pd.read_csv(V07_DIR / "validation_starts_timestamp_spaced.csv")
VALIDATION_STARTS = validation_start_df["start_idx"].astype(np.int64).to_numpy()
if len(VALIDATION_STARTS) < 6 or len(np.unique(VALIDATION_STARTS)) != len(VALIDATION_STARTS):
    raise RuntimeError("Frozen validation windows are missing or duplicated")
if "timestamp_ms" in validation_start_df:
    gaps = np.diff(validation_start_df["timestamp_ms"].to_numpy(np.int64)) / 60_000
    if len(gaps) and gaps.min() + 1e-9 < EVAL_EPISODE_MINUTES:
        raise RuntimeError("Frozen validation windows overlap")

protocol = dict(
    spec_id=SPEC_ID, authorization_spec=freeze.get("adjudication_spec_id"),
    labels=list(LABELS), selected_kappa=SELECTED_KAPPA,
    training_seeds=list(MASTER_SEEDS), primary_checkpoint=PRIMARY_CHECKPOINT,
    sensitivity_checkpoint=SENSITIVITY_CHECKPOINT,
    validation_starts=VALIDATION_STARTS.tolist(), loose_budget=LOOSE_BUDGET,
    strict_budget=STRICT_BUDGET, activity_floor_fraction=V08_CFG["activity_floor_fraction"],
    checkpoint_reselection_allowed=False, test_used=False,
)
atomic_csv(pd.DataFrame([protocol]), V08_DIR / "protocol.csv")
tmp = V08_DIR / "protocol.json.tmp"
with open(tmp, "w") as f: json.dump(protocol, f, indent=2)
os.replace(tmp, V08_DIR / "protocol.json")
display(validation_start_df)
print("authorization: PASS")
print("kappa:", SELECTED_KAPPA, "budgets:", LOOSE_BUDGET, STRICT_BUDGET)
print("training starts:", len(train_starts_all), "validation windows:", len(VALIDATION_STARTS))

,start_idx,timestamp_ms,timestamp_utc,minutes_since_previous
0,602028,1783692835905,2026-07-10 14:13:55.905000+00:00,NaN
1,619985,1783694635913,2026-07-10 14:43:55.913000+00:00,30.000133
2,637921,1783696435949,2026-07-10 15:13:55.949000+00:00,30.000600
3,655797,1783698236021,2026-07-10 15:43:56.021000+00:00,30.001200
4,673777,1783700036120,2026-07-10 16:13:56.120000+00:00,30.001650
5,691626,1783701836131,2026-07-10 16:43:56.131000+00:00,30.000183
6,709586,1783703636151,2026-07-10 17:13:56.151000+00:00,30.000333


authorization: PASS
kappa: 0.0131684977236533 budgets: 1.2142215137929977 0.7588884461206233
training starts: 415523 validation windows: 7


## Fixed-penalty TD3 and deterministic rollout

In [ ]:
def load_shared_online_weights(bundle, payload):
    bundle.se.load_state_dict(payload["state_encoder"])
    bundle.actor.load_state_dict(payload["actor"])
    bundle.reward_critics.load_state_dict(payload["reward_critics"])
    bundle.cost_critics.load_state_dict(payload["cost_critics"])
    bundle.tse.load_state_dict(bundle.se.state_dict())
    bundle.tactor.load_state_dict(bundle.actor.state_dict())
    bundle.treward_critics.load_state_dict(bundle.reward_critics.state_dict())
    bundle.tcost_critics.load_state_dict(bundle.cost_critics.state_dict())
    return bundle


def fixed_augmented_reward(reward, cost, fixed_penalty):
    return reward - float(fixed_penalty) * cost


def fixed_penalty_td_loss(bundle, batch, fixed_penalty):
    augmented_reward = fixed_augmented_reward(
        batch["reward"], batch["cost"], fixed_penalty
    )
    with torch.no_grad():
        next_state = bundle.tse(
            batch["next_market"], batch["next_signals"], batch["next_private"]
        )
        next_action = clipped_target_action(
            bundle.tactor,
            next_state,
            TRAIN_CFG["target_noise"],
            TRAIN_CFG["noise_clip"],
        )
        tq1, tq2 = bundle.treward_critics(next_state, next_action)
        target = (
            augmented_reward
            + TRAIN_CFG["gamma"]
            * (1.0 - batch["done"])
            * torch.minimum(tq1, tq2)
        )
    state = bundle.se(batch["market"], batch["signals"], batch["private"])
    q1, q2 = bundle.reward_critics(state, batch["action"])
    loss = F.mse_loss(q1, target) + F.mse_loss(q2, target)
    return loss, float(
        (float(fixed_penalty) * batch["cost"]).mean().detach().cpu()
    )


def update_fixed_penalty_td3(
    bundle, replay, data, signals, global_gradient_step, expert, fixed_penalty
):
    online = reconstruct_batch(
        data, signals, replay.sample(TRAIN_CFG["batch_size"])
    )
    (
        e_state_idx, e_next_idx, e_private, e_next_private,
        e_action, e_reward, e_cost, e_done,
    ) = expert.sample_critic(IMM_CFG["expert_critic_batch_size"])
    expert_batch = reconstruct_batch(data, signals, dict(
        state_idx=e_state_idx,
        next_state_idx=e_next_idx,
        private=e_private,
        next_private=e_next_private,
        action=e_action,
        reward=e_reward,
        cost=e_cost,
        done=e_done,
    ))

    bundle.se_opt.zero_grad(set_to_none=True)
    bundle.reward_critics_opt.zero_grad(set_to_none=True)
    online_loss, online_penalty = fixed_penalty_td_loss(
        bundle, online, fixed_penalty
    )
    expert_loss, expert_penalty = fixed_penalty_td_loss(
        bundle, expert_batch, fixed_penalty
    )
    critic_loss = 0.5 * online_loss + 0.5 * expert_loss
    critic_loss.backward()
    torch.nn.utils.clip_grad_norm_(bundle.se.parameters(), 10.0)
    torch.nn.utils.clip_grad_norm_(bundle.reward_critics.parameters(), 10.0)
    bundle.se_opt.step()
    bundle.reward_critics_opt.step()

    actor_updated = global_gradient_step % TRAIN_CFG["policy_delay"] == 0
    metrics = dict(
        actor_updated=bool(actor_updated),
        actor_loss=np.nan,
        rl_actor_loss=np.nan,
        bc_loss=np.nan,
        bc_weight=np.nan,
    )
    if actor_updated:
        set_grad(bundle.reward_critics, False)
        bundle.se_opt.zero_grad(set_to_none=True)
        bundle.actor_opt.zero_grad(set_to_none=True)
        state_t = bundle.se(
            online["market"], online["signals"], online["private"]
        )
        policy_action = bundle.actor(state_t)
        rl_loss = -bundle.reward_critics.q1(state_t, policy_action).mean()

        e_idx, e_private_actor, e_action_actor = expert.sample_actor(
            IMM_CFG["expert_actor_batch_size"]
        )
        e_market, e_signals, e_private_t = tensor_state(
            data, signals, e_idx, e_private_actor
        )
        e_state = bundle.se(e_market, e_signals, e_private_t)
        bc_loss = F.mse_loss(
            bundle.actor(e_state),
            torch.from_numpy(e_action_actor).to(DEVICE),
        )
        bc_weight = bc_weight_at(global_gradient_step)
        actor_loss = rl_loss + bc_weight * bc_loss
        actor_loss.backward()
        torch.nn.utils.clip_grad_norm_(bundle.se.parameters(), 10.0)
        torch.nn.utils.clip_grad_norm_(bundle.actor.parameters(), 10.0)
        bundle.se_opt.step()
        bundle.actor_opt.step()
        set_grad(bundle.reward_critics, True)

        polyak_update(bundle.tse, bundle.se, TRAIN_CFG["tau"])
        polyak_update(bundle.tactor, bundle.actor, TRAIN_CFG["tau"])
        polyak_update(
            bundle.treward_critics, bundle.reward_critics, TRAIN_CFG["tau"]
        )
        metrics.update(
            actor_loss=float(actor_loss.detach().cpu()),
            rl_actor_loss=float(rl_loss.detach().cpu()),
            bc_loss=float(bc_loss.detach().cpu()),
            bc_weight=float(bc_weight),
        )

    metrics.update(
        reward_critic_loss=float(critic_loss.detach().cpu()),
        online_fixed_penalty=float(online_penalty),
        expert_fixed_penalty=float(expert_penalty),
        fixed_penalty=float(fixed_penalty),
    )
    return metrics


@torch.no_grad()
def rollout_policy(
    bundle, start_idx, split_name, label, fixed_penalty, checkpoint_episode
):
    env = make_env(data, signals, 0, EVAL_EPISODE_MINUTES)
    obs, _ = env.reset(
        split_name=split_name,
        start_idx=int(start_idx),
        require_full_duration=True,
    )
    gamma_power = 1.0
    discounted_cost = 0.0
    reward_sum = 0.0
    pnl_sum = 0.0
    filled_notional_sum = 0.0
    transitions = []
    step_count = 0
    terminal_reason = "unknown"
    start_timestamp_ms = int(data.timestamp_ms[int(start_idx)])
    final_timestamp_ms = start_timestamp_ms

    while True:
        action = get_action(bundle, obs, noise=0.0)
        next_obs, _, terminated, truncated, info = env.step(action)
        next_idx = int(info["state_idx_end"])
        signal_terminal = bool(not ready[next_idx])
        cost, cost_parts = RISK_COST_FN.compute(
            inventory=float(info["inventory"]),
            quote_size=float(data.quote_size),
        )
        reward, _ = ECONOMIC_REWARD_FN.compute(
            pnl=float(info["pnl"]),
            filled_notional=float(info["filled_notional"]),
            inventory=float(info["inventory"]),
            quote_size=float(data.quote_size),
            mid_price=float(data.mid_price[next_idx]),
        )
        offgrid_after = int(
            info["offgrid_live_child_count_after_auto_cancel"]
        )
        if offgrid_after != 0:
            raise AssertionError("off-grid order survived evaluation boundary")

        step_count += 1
        final_timestamp_ms = int(info["timestamp_end_ms"])
        discounted_cost += gamma_power * float(cost)
        gamma_power *= float(RISK_CFG["gamma_cost"])
        reward_sum += float(reward)
        pnl_sum += float(info["pnl"])
        filled_notional_sum += float(info["filled_notional"])
        row = dict(
            symbol=SYMBOL,
            split=split_name,
            label=label,
            fixed_penalty=float(fixed_penalty),
            checkpoint_episode=int(checkpoint_episode),
            start_idx=int(start_idx),
            step=step_count,
            reward=float(reward),
            pnl=float(info["pnl"]),
            cost=float(cost),
            inventory_N=float(cost_parts["inventory_N"]),
            filled_notional=float(info["filled_notional"]),
            offgrid_after=offgrid_after,
        )
        for j, value in enumerate(action):
            row[f"action_{j}"] = float(value)
        transitions.append(row)

        if signal_terminal:
            terminal_reason = "signal_boundary"
            break
        if truncated:
            terminal_reason = "time_limit"
            break
        if terminated:
            terminal_reason = "data_end"
            break
        obs = next_obs

    elapsed_minutes = (
        final_timestamp_ms - start_timestamp_ms
    ) / 60_000.0
    episode = dict(
        symbol=SYMBOL,
        split=split_name,
        label=label,
        fixed_penalty=float(fixed_penalty),
        checkpoint_episode=int(checkpoint_episode),
        start_idx=int(start_idx),
        steps=step_count,
        elapsed_minutes=float(elapsed_minutes),
        horizon_completed=bool(
            elapsed_minutes >= 0.95 * EVAL_EPISODE_MINUTES
        ),
        terminal_reason=terminal_reason,
        normalized_discounted_cost=float(
            (1.0 - RISK_CFG["gamma_cost"]) * discounted_cost
        ),
        reward_sum=float(reward_sum),
        pnl_sum=float(pnl_sum),
        filled_notional=float(filled_notional_sum),
    )
    return episode, pd.DataFrame(transitions)


## Seed-isolated resumable checkpoints

In [ ]:
@dataclass
class V08TrainingState:
    bundle: RiskBundle
    replay: RiskIndexedReplayBuffer
    global_env_step: int
    global_gradient_step: int
    branch_env_step: int = 0
    branch_gradient_step: int = 0
    completed_episodes: int = 0
    initial_fingerprint: str = ""
    imported_from: str = ""

def replay_to_payload(replay):
    n=int(replay.size)
    return dict(capacity=int(replay.capacity), private_dim=int(replay.private_dim),
        action_dim=int(replay.action_dim), pos=int(replay.pos), size=n,
        rng_state=copy.deepcopy(replay.rng.bit_generator.state),
        state_idx=replay.state_idx[:n].copy(), next_state_idx=replay.next_state_idx[:n].copy(),
        private=replay.private[:n].copy(), next_private=replay.next_private[:n].copy(),
        action=replay.action[:n].copy(), reward=replay.reward[:n].copy(),
        cost=replay.cost[:n].copy(), done=replay.done[:n].copy())

def replay_from_payload(p):
    r=RiskIndexedReplayBuffer(int(p["capacity"]), private_dim=int(p["private_dim"]),
        action_dim=int(p["action_dim"]), seed=0)
    n=int(p["size"])
    for name in ["state_idx","next_state_idx","private","next_private","action","reward","cost","done"]:
        getattr(r,name)[:n]=p[name]
    r.size=n; r.pos=int(p["pos"]); r.rng.bit_generator.state=p["rng_state"]
    return r

def bundle_to_payload(b):
    return dict(state_encoder=b.se.state_dict(), actor=b.actor.state_dict(),
        reward_critics=b.reward_critics.state_dict(), cost_critics=b.cost_critics.state_dict(),
        target_state_encoder=b.tse.state_dict(), target_actor=b.tactor.state_dict(),
        target_reward_critics=b.treward_critics.state_dict(), target_cost_critics=b.tcost_critics.state_dict(),
        state_encoder_optimizer=b.se_opt.state_dict(), actor_optimizer=b.actor_opt.state_dict(),
        reward_critics_optimizer=b.reward_critics_opt.state_dict(), cost_critics_optimizer=b.cost_critics_opt.state_dict())

def optimizer_to_device(opt,device):
    for st in opt.state.values():
        for k,v in st.items():
            if torch.is_tensor(v): st[k]=v.to(device)

def bundle_from_payload(p):
    b=make_bundle(TRAIN_CFG)
    b.se.load_state_dict(p["state_encoder"]); b.actor.load_state_dict(p["actor"])
    b.reward_critics.load_state_dict(p["reward_critics"]); b.cost_critics.load_state_dict(p["cost_critics"])
    b.tse.load_state_dict(p["target_state_encoder"]); b.tactor.load_state_dict(p["target_actor"])
    b.treward_critics.load_state_dict(p["target_reward_critics"]); b.tcost_critics.load_state_dict(p["target_cost_critics"])
    b.se_opt.load_state_dict(p["state_encoder_optimizer"]); b.actor_opt.load_state_dict(p["actor_optimizer"])
    b.reward_critics_opt.load_state_dict(p["reward_critics_optimizer"]); b.cost_critics_opt.load_state_dict(p["cost_critics_optimizer"])
    for o in [b.se_opt,b.actor_opt,b.reward_critics_opt,b.cost_critics_opt]: optimizer_to_device(o,DEVICE)
    return b

def branch_dir(seed,label):
    p=V08_DIR/"seeds"/f"seed_{int(seed)}"/label.lower(); p.mkdir(parents=True,exist_ok=True); return p
def latest_checkpoint_path(seed,label): return branch_dir(seed,label)/"checkpoints"/"latest.pt"
def snapshot_checkpoint_path(seed,label,episode): return branch_dir(seed,label)/"checkpoints"/f"episode_{int(episode):02d}.pt"
def training_part_paths(seed,label,episode):
    p=branch_dir(seed,label)/"training_parts"; p.mkdir(parents=True,exist_ok=True); stem=f"episode_{int(episode):02d}"
    return p/f"{stem}_summary.csv",p/f"{stem}_env.parquet",p/f"{stem}_loss.parquet"

def save_state_checkpoint(seed,label,state,env_rng,expert):
    payload=dict(spec_id=SPEC_ID,seed=int(seed),label=label,fixed_penalty=float(PENALTY_GRID[label]),
        v08_config=V08_CFG,train_config=TRAIN_CFG,bundle=bundle_to_payload(state.bundle),replay=replay_to_payload(state.replay),
        global_env_step=int(state.global_env_step),global_gradient_step=int(state.global_gradient_step),
        branch_env_step=int(state.branch_env_step),branch_gradient_step=int(state.branch_gradient_step),
        completed_episodes=int(state.completed_episodes),initial_fingerprint=state.initial_fingerprint,
        imported_from=state.imported_from,env_rng_state=copy.deepcopy(env_rng.bit_generator.state),
        expert_rng_state=copy.deepcopy(expert.rng.bit_generator.state),numpy_random_state=np.random.get_state(),
        python_random_state=random.getstate(),torch_cpu_rng_state=torch.get_rng_state(),
        torch_cuda_rng_state=torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None)
    atomic_torch_save(payload,latest_checkpoint_path(seed,label))
    if state.completed_episodes in set(V08_CFG["snapshot_episodes"]):
        snap=snapshot_checkpoint_path(seed,label,state.completed_episodes)
        if not snap.exists(): atomic_torch_save(payload,snap)

def restore_rng(p,env_rng,expert):
    env_rng.bit_generator.state=p["env_rng_state"]; expert.rng.bit_generator.state=p["expert_rng_state"]
    np.random.set_state(p["numpy_random_state"]); random.setstate(p["python_random_state"])
    torch.set_rng_state(p["torch_cpu_rng_state"].cpu())
    if torch.cuda.is_available() and p.get("torch_cuda_rng_state") is not None: torch.cuda.set_rng_state_all(p["torch_cuda_rng_state"])

def state_from_payload(p):
    return V08TrainingState(bundle=bundle_from_payload(p["bundle"]),replay=replay_from_payload(p["replay"]),
        global_env_step=int(p["global_env_step"]),global_gradient_step=int(p["global_gradient_step"]),
        branch_env_step=int(p["branch_env_step"]),branch_gradient_step=int(p["branch_gradient_step"]),
        completed_episodes=int(p["completed_episodes"]),initial_fingerprint=str(p.get("initial_fingerprint","")),
        imported_from=str(p.get("imported_from","")))

def init_or_load_branch(seed,label):
    if int(seed) not in MASTER_SEEDS or label not in LABELS: raise KeyError((seed,label))
    branch_seed=int(seed); env_rng=np.random.default_rng(branch_seed)
    expert=RiskExpertSampler(data,demos,ECONOMIC_REWARD_FN,RISK_COST_FN,branch_seed)
    latest=latest_checkpoint_path(seed,label)
    if latest.exists():
        p=torch_load_compat(latest)
        if p.get("spec_id")!=SPEC_ID or int(p.get("seed"))!=branch_seed or p.get("label")!=label: raise RuntimeError("checkpoint identity mismatch")
        if not np.isclose(float(p["fixed_penalty"]),PENALTY_GRID[label],rtol=0,atol=1e-12): raise RuntimeError("checkpoint penalty mismatch")
        state=state_from_payload(p); restore_rng(p,env_rng,expert); return state,env_rng,expert
    torch.manual_seed(branch_seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(branch_seed)
    np.random.seed(branch_seed); random.seed(branch_seed)
    shared=torch_load_compat(V04_DIR/"shared_pretrain_checkpoint.pt")
    bundle=load_shared_online_weights(make_bundle(TRAIN_CFG),shared)
    state=V08TrainingState(bundle=bundle,replay=RiskIndexedReplayBuffer(TRAIN_CFG["replay_capacity"],seed=branch_seed),
        global_env_step=int(shared.get("global_env_step",20000)),global_gradient_step=int(shared.get("global_gradient_step",7500)),
        initial_fingerprint=module_digest(bundle.se,bundle.actor,bundle.reward_critics),imported_from=str(V04_DIR/"shared_pretrain_checkpoint.pt"))
    save_state_checkpoint(seed,label,state,env_rng,expert); return state,env_rng,expert


## Resumable matched training

In [ ]:
def run_penalty_branch_until(seed, label, target_episodes):
    target_episodes = min(
        int(target_episodes), V08_CFG["target_episodes"]
    )
    state, env_rng, expert = init_or_load_branch(seed, label)
    fixed_penalty = float(PENALTY_GRID[label])

    if state.completed_episodes >= target_episodes:
        print(
            f"{label}: already complete through episode "
            f"{state.completed_episodes}"
        )
        return

    env = make_env(data, signals, int(seed) + 1, TRAIN_EPISODE_MINUTES)
    started = time.perf_counter()
    while state.completed_episodes < target_episodes:
        episode_number = state.completed_episodes + 1
        start_idx = int(env_rng.choice(train_starts_all))
        obs, _ = env.reset(
            split_name=TRAIN_SPLIT,
            start_idx=start_idx,
            require_full_duration=True,
        )
        gamma_power = 1.0
        discounted_cost = 0.0
        episode_reward = 0.0
        episode_pnl = 0.0
        episode_penalty = 0.0
        episode_steps = 0
        terminal_reason = "unknown"
        env_rows = []
        loss_rows = []

        while True:
            state.global_env_step += 1
            state.branch_env_step += 1
            episode_steps += 1
            state_idx = int(env.idx)
            private = np.asarray(obs["private"], np.float32).copy()
            action = get_action(
                state.bundle, obs, TRAIN_CFG["exploration_noise"]
            )
            next_obs, _, terminated, truncated, info = env.step(action)
            next_idx = int(info["state_idx_end"])
            signal_terminal = bool(not ready[next_idx])
            done = bool(signal_terminal or terminated or truncated)
            if int(info["offgrid_live_child_count_after_auto_cancel"]) != 0:
                raise AssertionError("off-grid order reached next policy action")

            reward, _ = ECONOMIC_REWARD_FN.compute(
                pnl=float(info["pnl"]),
                filled_notional=float(info["filled_notional"]),
                inventory=float(info["inventory"]),
                quote_size=float(data.quote_size),
                mid_price=float(data.mid_price[next_idx]),
            )
            cost, cost_parts = RISK_COST_FN.compute(
                inventory=float(info["inventory"]),
                quote_size=float(data.quote_size),
            )
            next_private = (
                np.zeros(21, np.float32)
                if done
                else np.asarray(next_obs["private"], np.float32).copy()
            )
            state.replay.add(
                state_idx=state_idx,
                next_state_idx=next_idx,
                private=private,
                next_private=next_private,
                action=action,
                reward=reward,
                cost=cost,
                done=done,
            )
            discounted_cost += gamma_power * float(cost)
            gamma_power *= float(RISK_CFG["gamma_cost"])
            episode_reward += float(reward)
            episode_pnl += float(info["pnl"])
            episode_penalty += fixed_penalty * float(cost)
            env_rows.append(dict(
                seed=int(seed),
                label=label,
                episode=episode_number,
                start_idx=start_idx,
                branch_env_step=state.branch_env_step,
                global_env_step=state.global_env_step,
                reward=float(reward),
                pnl=float(info["pnl"]),
                cost=float(cost),
                inventory_N=float(cost_parts["inventory_N"]),
                fixed_penalty=fixed_penalty,
                augmented_reward=float(reward - fixed_penalty * cost),
                offgrid_after=int(
                    info["offgrid_live_child_count_after_auto_cancel"]
                ),
            ))

            if (
                len(state.replay) >= TRAIN_CFG["batch_size"]
                and state.branch_env_step % TRAIN_CFG["update_every"] == 0
            ):
                state.global_gradient_step += 1
                state.branch_gradient_step += 1
                metrics = update_fixed_penalty_td3(
                    state.bundle,
                    state.replay,
                    data,
                    signals,
                    state.global_gradient_step,
                    expert,
                    fixed_penalty,
                )
                loss_rows.append(dict(
                    label=label,
                    episode=episode_number,
                    branch_env_step=state.branch_env_step,
                    global_env_step=state.global_env_step,
                    global_gradient_step=state.global_gradient_step,
                    branch_gradient_step=state.branch_gradient_step,
                    **metrics,
                ))

            if done:
                terminal_reason = (
                    "signal_boundary" if signal_terminal else
                    "time_limit" if truncated else "data_end"
                )
                break
            obs = next_obs

        state.completed_episodes += 1
        episode_row = dict(
            seed=int(seed),
            label=label,
            episode=state.completed_episodes,
            start_idx=start_idx,
            steps=episode_steps,
            terminal_reason=terminal_reason,
            mc_risk=float(
                (1.0 - RISK_CFG["gamma_cost"]) * discounted_cost
            ),
            fixed_penalty=fixed_penalty,
            reward_sum=episode_reward,
            pnl_sum=episode_pnl,
            penalty_sum=episode_penalty,
            augmented_reward_sum=episode_reward - episode_penalty,
        )
        summary_path, env_path, loss_path = training_part_paths(
            seed, label, state.completed_episodes
        )
        atomic_csv(pd.DataFrame([episode_row]), summary_path)
        atomic_parquet(pd.DataFrame(env_rows), env_path)
        atomic_parquet(pd.DataFrame(loss_rows), loss_path)
        save_state_checkpoint(seed, label, state, env_rng, expert)

    print(
        f"{label}: reached {state.completed_episodes} episodes in "
        f"{(time.perf_counter() - started) / 60:.1f} min"
    )


In [ ]:
# Preflight: both arms within every seed originate from the same frozen weights and RNG stream.
rows=[]
for seed in MASTER_SEEDS:
    digests=[]
    for label in LABELS:
        state,env_rng,_=init_or_load_branch(seed,label)
        digests.append(state.initial_fingerprint)
        rng_digest=hashlib.sha256(json.dumps(env_rng.bit_generator.state,sort_keys=True).encode()).hexdigest()
        rows.append(dict(seed=seed,label=label,episode=state.completed_episodes,initial_fingerprint=state.initial_fingerprint,env_rng_digest=rng_digest))
        del state
    if len(set(digests)) != 1: raise RuntimeError(f"Arm initialization mismatch for seed {seed}")
preflight_df=pd.DataFrame(rows)
fresh=preflight_df[preflight_df.episode.eq(0)]
if len(fresh) and fresh.groupby("seed")["env_rng_digest"].nunique().ne(1).any(): raise RuntimeError("Fresh arms do not share the same seed RNG")
display(preflight_df); atomic_csv(preflight_df,V08_DIR/"preflight_initialization.csv")

,seed,label,episode,initial_fingerprint,env_rng_digest
0,10801,K0_UNCONSTRAINED,10,2c40b93a485d84f98afb470fa7ee6bd8a934be70a2c0c0...,18ac9500a49ff2e91585d7b1d4019bc73d949145838e61...
1,10801,K4_VERY_STRONG,10,2c40b93a485d84f98afb470fa7ee6bd8a934be70a2c0c0...,18ac9500a49ff2e91585d7b1d4019bc73d949145838e61...
2,10802,K0_UNCONSTRAINED,10,2c40b93a485d84f98afb470fa7ee6bd8a934be70a2c0c0...,55f951512766152b6bc8a28e154f988f83f00643c6ef0c...
3,10802,K4_VERY_STRONG,10,2c40b93a485d84f98afb470fa7ee6bd8a934be70a2c0c0...,55f951512766152b6bc8a28e154f988f83f00643c6ef0c...
4,10803,K0_UNCONSTRAINED,10,2c40b93a485d84f98afb470fa7ee6bd8a934be70a2c0c0...,832b8b0a2c8343d487038bc5843148320b8125d8462741...
5,10803,K4_VERY_STRONG,10,2c40b93a485d84f98afb470fa7ee6bd8a934be70a2c0c0...,832b8b0a2c8343d487038bc5843148320b8125d8462741...
6,10804,K0_UNCONSTRAINED,10,2c40b93a485d84f98afb470fa7ee6bd8a934be70a2c0c0...,6303fc728249310b20f552ff595394ec73ee6e4d95806e...
7,10804,K4_VERY_STRONG,10,2c40b93a485d84f98afb470fa7ee6bd8a934be70a2c0c0...,6303fc728249310b20f552ff595394ec73ee6e4d95806e...
8,10805,K0_UNCONSTRAINED,10,2c40b93a485d84f98afb470fa7ee6bd8a934be70a2c0c0...,3449c17e90048238f1e01018fc8128988019c0d166056a...
9,10805,K4_VERY_STRONG,10,2c40b93a485d84f98afb470fa7ee6bd8a934be70a2c0c0...,3449c17e90048238f1e01018fc8128988019c0d166056a...


### Seed 10801: train both matched arms through primary checkpoint 10

In [ ]:
run_penalty_branch_until(10801, "K0_UNCONSTRAINED", 10)
run_penalty_branch_until(10801, "K4_VERY_STRONG", 10)

### Seed 10802: train both matched arms through primary checkpoint 10

In [ ]:
run_penalty_branch_until(10802, "K0_UNCONSTRAINED", 10)
run_penalty_branch_until(10802, "K4_VERY_STRONG", 10)

### Seed 10803: train both matched arms through primary checkpoint 10

In [ ]:
run_penalty_branch_until(10803, "K0_UNCONSTRAINED", 10)
run_penalty_branch_until(10803, "K4_VERY_STRONG", 10)

### Seed 10804: train both matched arms through primary checkpoint 10

In [ ]:
run_penalty_branch_until(10804, "K0_UNCONSTRAINED", 10)
run_penalty_branch_until(10804, "K4_VERY_STRONG", 10)

### Seed 10805: train both matched arms through primary checkpoint 10

In [ ]:
run_penalty_branch_until(10805, "K0_UNCONSTRAINED", 10)
run_penalty_branch_until(10805, "K4_VERY_STRONG", 10)

## Primary-checkpoint training integrity

In [ ]:
training_rows=[]
for seed in MASTER_SEEDS:
    starts={}
    for label in LABELS:
        p=snapshot_checkpoint_path(seed,label,10)
        if not p.exists(): raise FileNotFoundError(p)
        payload=torch_load_compat(p)
        if int(payload["completed_episodes"])!=10: raise RuntimeError("episode-10 snapshot mismatch")
        vals=[]
        for ep in range(1,11): vals.append(int(pd.read_csv(training_part_paths(seed,label,ep)[0]).iloc[0]["start_idx"]))
        starts[label]=vals
        training_rows.append(dict(seed=seed,label=label,checkpoint=10,initial_fingerprint=payload["initial_fingerprint"],starts_digest=hashlib.sha256(np.asarray(vals,np.int64).tobytes()).hexdigest()))
    if starts[LABELS[0]] != starts[LABELS[1]]: raise RuntimeError(f"Unmatched training starts for seed {seed}")
training_integrity_df=pd.DataFrame(training_rows)
if training_integrity_df.groupby("seed")["initial_fingerprint"].nunique().ne(1).any(): raise RuntimeError("Arm fingerprints differ")
if training_integrity_df["initial_fingerprint"].nunique()!=1: raise RuntimeError("All seeds must use the same frozen pre-branch weights")
if training_integrity_df.groupby("label")["starts_digest"].nunique().lt(3).any(): raise RuntimeError("Fewer than three distinct seed-specific training schedules")
display(training_integrity_df); atomic_csv(training_integrity_df,V08_DIR/"training_integrity_ep10.csv")

,seed,label,checkpoint,initial_fingerprint,starts_digest
0,10801,K0_UNCONSTRAINED,10,2c40b93a485d84f98afb470fa7ee6bd8a934be70a2c0c0...,892e5fc7f4c7f7301a7b163f3730374e9a14238c4fab53...
1,10801,K4_VERY_STRONG,10,2c40b93a485d84f98afb470fa7ee6bd8a934be70a2c0c0...,892e5fc7f4c7f7301a7b163f3730374e9a14238c4fab53...
2,10802,K0_UNCONSTRAINED,10,2c40b93a485d84f98afb470fa7ee6bd8a934be70a2c0c0...,fa24131bf7ca490ca7da6da48b6e0dcd011f745d62a0e4...
3,10802,K4_VERY_STRONG,10,2c40b93a485d84f98afb470fa7ee6bd8a934be70a2c0c0...,fa24131bf7ca490ca7da6da48b6e0dcd011f745d62a0e4...
4,10803,K0_UNCONSTRAINED,10,2c40b93a485d84f98afb470fa7ee6bd8a934be70a2c0c0...,e6aa8435905e9d3bfdba3e421efc27adf94f9b8f3539d7...
5,10803,K4_VERY_STRONG,10,2c40b93a485d84f98afb470fa7ee6bd8a934be70a2c0c0...,e6aa8435905e9d3bfdba3e421efc27adf94f9b8f3539d7...
6,10804,K0_UNCONSTRAINED,10,2c40b93a485d84f98afb470fa7ee6bd8a934be70a2c0c0...,1c6ee12ab41be174a559fea18e6b9d92145fcb17774dcb...
7,10804,K4_VERY_STRONG,10,2c40b93a485d84f98afb470fa7ee6bd8a934be70a2c0c0...,1c6ee12ab41be174a559fea18e6b9d92145fcb17774dcb...
8,10805,K0_UNCONSTRAINED,10,2c40b93a485d84f98afb470fa7ee6bd8a934be70a2c0c0...,6720e2bf5431aa277d1a2cdaf28394f583a209939cd14c...
9,10805,K4_VERY_STRONG,10,2c40b93a485d84f98afb470fa7ee6bd8a934be70a2c0c0...,6720e2bf5431aa277d1a2cdaf28394f583a209939cd14c...


## Seed-aware resumable validation

In [ ]:
def evaluation_part_paths(seed,stage,label,checkpoint_episode,start_idx):
    folder=branch_dir(seed,label)/"evaluation_parts"/stage/f"episode_{int(checkpoint_episode):02d}"
    folder.mkdir(parents=True,exist_ok=True)
    return folder/f"start_{int(start_idx)}_episode.csv",folder/f"start_{int(start_idx)}_transitions.parquet"

def load_snapshot_bundle(seed,label,checkpoint_episode):
    p=snapshot_checkpoint_path(seed,label,checkpoint_episode)
    if not p.exists(): raise FileNotFoundError(p)
    payload=torch_load_compat(p)
    if payload.get("spec_id")!=SPEC_ID or int(payload["seed"])!=int(seed) or payload["label"]!=label: raise RuntimeError("snapshot identity mismatch")
    if int(payload["completed_episodes"])!=int(checkpoint_episode): raise RuntimeError("snapshot episode mismatch")
    b=bundle_from_payload(payload["bundle"]); b.se.eval(); b.actor.eval(); b.reward_critics.eval(); return b

def evaluate_until(seed,stage,label,checkpoint_episode,starts):
    b=load_snapshot_bundle(seed,label,checkpoint_episode)
    for start_idx in np.asarray(starts,np.int64):
        ep_path,tr_path=evaluation_part_paths(seed,stage,label,checkpoint_episode,start_idx)
        if ep_path.exists() and tr_path.exists(): continue
        episode,tr=rollout_policy(b,int(start_idx),VALIDATION_SPLIT,label,PENALTY_GRID[label],checkpoint_episode)
        episode["seed"]=int(seed); tr.insert(0,"seed",int(seed))
        atomic_csv(pd.DataFrame([episode]),ep_path); atomic_parquet(tr,tr_path)
    del b; gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

def load_evaluations(seed,stage,label,checkpoint_episode,starts):
    eps=[]; trs=[]
    for st in starts:
        e,t=evaluation_part_paths(seed,stage,label,checkpoint_episode,int(st)); eps.append(pd.read_csv(e)); trs.append(pd.read_parquet(t))
    return pd.concat(eps,ignore_index=True),pd.concat(trs,ignore_index=True)


In [ ]:
evaluate_until(10801, "validation_primary_h10", "K0_UNCONSTRAINED", 10, VALIDATION_STARTS)
evaluate_until(10801, "validation_primary_h10", "K4_VERY_STRONG", 10, VALIDATION_STARTS)

In [ ]:
evaluate_until(10802, "validation_primary_h10", "K0_UNCONSTRAINED", 10, VALIDATION_STARTS)
evaluate_until(10802, "validation_primary_h10", "K4_VERY_STRONG", 10, VALIDATION_STARTS)

In [ ]:
evaluate_until(10803, "validation_primary_h10", "K0_UNCONSTRAINED", 10, VALIDATION_STARTS)
evaluate_until(10803, "validation_primary_h10", "K4_VERY_STRONG", 10, VALIDATION_STARTS)

In [ ]:
evaluate_until(10804, "validation_primary_h10", "K0_UNCONSTRAINED", 10, VALIDATION_STARTS)
evaluate_until(10804, "validation_primary_h10", "K4_VERY_STRONG", 10, VALIDATION_STARTS)

In [ ]:
evaluate_until(10805, "validation_primary_h10", "K0_UNCONSTRAINED", 10, VALIDATION_STARTS)
evaluate_until(10805, "validation_primary_h10", "K4_VERY_STRONG", 10, VALIDATION_STARTS)

In [ ]:
episode_frames = []
transition_frames = []

for seed in MASTER_SEEDS:
    for label in LABELS:
        episodes, transitions = load_evaluations(
            seed=seed,
            stage="validation_primary_h10",
            label=label,
            checkpoint_episode=10,
            starts=VALIDATION_STARTS,
        )
        episodes["checkpoint_episode"] = 10
        transitions["checkpoint_episode"] = 10
        episode_frames.append(episodes)
        transition_frames.append(transitions)

episodes_df = pd.concat(episode_frames, ignore_index=True)
transitions_df = pd.concat(transition_frames, ignore_index=True)

expected_rows = len(MASTER_SEEDS) * len(LABELS) * len(VALIDATION_STARTS)

if len(episodes_df) != expected_rows:
    raise RuntimeError(
        f"Expected {expected_rows} episode rows, got {len(episodes_df)}"
    )

if episodes_df.duplicated(
    ["seed", "label", "checkpoint_episode", "start_idx"]
).any():
    raise RuntimeError("Duplicate evaluation cells found")

if not episodes_df["horizon_completed"].astype(bool).all():
    raise RuntimeError("At least one evaluation horizon is incomplete")

if int(transitions_df["offgrid_after"].max()) != 0:
    raise RuntimeError("Off-grid invariant failed")


def hierarchical_bootstrap(values_by_seed, n_boot, rng):
    seeds = sorted(values_by_seed)
    draws = np.empty(n_boot, dtype=float)

    for bootstrap_idx in range(n_boot):
        sampled_seeds = rng.choice(
            seeds,
            size=len(seeds),
            replace=True,
        )

        sampled_seed_means = []

        for seed in sampled_seeds:
            values = np.asarray(
                values_by_seed[int(seed)],
                dtype=float,
            )

            sampled_windows = rng.choice(
                values,
                size=len(values),
                replace=True,
            )

            sampled_seed_means.append(sampled_windows.mean())

        draws[bootstrap_idx] = np.mean(sampled_seed_means)

    return (
        float(np.quantile(draws, 0.025)),
        float(np.quantile(draws, 0.975)),
    )


seed_effect_rows = []
comparison_rows = []

for metric in ["normalized_discounted_cost", "pnl_sum"]:
    differences_by_seed = {}

    for seed in MASTER_SEEDS:
        seed_episodes = episodes_df[
            episodes_df["seed"].eq(seed)
        ]

        paired = seed_episodes.pivot(
            index="start_idx",
            columns="label",
            values=metric,
        ).loc[VALIDATION_STARTS]

        differences = (
            paired["K4_VERY_STRONG"]
            - paired["K0_UNCONSTRAINED"]
        ).to_numpy(float)

        differences_by_seed[int(seed)] = differences

        seed_effect_rows.append(
            {
                "seed": int(seed),
                "checkpoint_episode": 10,
                "metric": metric,
                "mean_difference": float(differences.mean()),
                "k4_better_windows": int((differences < 0).sum()),
                "windows": int(len(differences)),
            }
        )

    point_estimate = float(
        np.mean([
            values.mean()
            for values in differences_by_seed.values()
        ])
    )

    ci_low, ci_high = hierarchical_bootstrap(
        differences_by_seed,
        n_boot=V08_CFG["bootstrap_samples"],
        rng=np.random.default_rng(
            SEED
            + (0 if metric == "normalized_discounted_cost" else 1000)
        ),
    )

    comparison_rows.append(
        {
            "checkpoint_episode": 10,
            "endpoint_role": "PRIMARY",
            "metric": metric,
            "mean_difference": point_estimate,
            "ci_low": ci_low,
            "ci_high": ci_high,
            "seeds": len(differences_by_seed),
            "windows_per_seed": len(VALIDATION_STARTS),
        }
    )

seed_effects_df = pd.DataFrame(seed_effect_rows)
comparison_df = pd.DataFrame(comparison_rows)

summary_rows = []

for label in LABELS:
    label_episodes = episodes_df[
        episodes_df["label"].eq(label)
    ]
    label_transitions = transitions_df[
        transitions_df["label"].eq(label)
    ]

    absolute_inventory = label_transitions["inventory_N"].abs()
    action_columns = [f"action_{j}" for j in range(4)]

    summary_rows.append(
        {
            "checkpoint_episode": 10,
            "label": label,
            "risk": float(
                label_episodes[
                    "normalized_discounted_cost"
                ].mean()
            ),
            "mean_pnl": float(
                label_episodes["pnl_sum"].mean()
            ),
            "mean_filled_notional": float(
                label_episodes["filled_notional"].mean()
            ),
            "mean_abs_inventory": float(
                absolute_inventory.mean()
            ),
            "p95_abs_inventory": float(
                absolute_inventory.quantile(0.95)
            ),
            "band_violation": float(
                (
                    absolute_inventory
                    > RISK_CFG["reference_band_N"]
                ).mean()
            ),
            "action_saturation": float(
                (
                    label_transitions[action_columns].abs()
                    > 0.98
                ).to_numpy().mean()
            ),
        }
    )

summary_df = pd.DataFrame(summary_rows)

primary_risk = comparison_df[
    comparison_df["metric"].eq(
        "normalized_discounted_cost"
    )
].iloc[0]

summary_index = summary_df.set_index("label")

activity_ratio = float(
    summary_index.loc[
        "K4_VERY_STRONG",
        "mean_filled_notional",
    ]
    / summary_index.loc[
        "K0_UNCONSTRAINED",
        "mean_filled_notional",
    ]
)

k4_episodes = episodes_df[
    episodes_df["label"].eq("K4_VERY_STRONG")
]

k4_risk_by_seed = {
    int(seed): frame.set_index("start_idx").loc[
        VALIDATION_STARTS,
        "normalized_discounted_cost",
    ].to_numpy(float)
    for seed, frame in k4_episodes.groupby("seed")
}

k4_risk = float(
    np.mean([
        values.mean()
        for values in k4_risk_by_seed.values()
    ])
)

k4_risk_ci_low, k4_risk_ci_high = hierarchical_bootstrap(
    k4_risk_by_seed,
    n_boot=V08_CFG["bootstrap_samples"],
    rng=np.random.default_rng(SEED + 777),
)

risk_reduction_confirmed = bool(
    primary_risk["ci_high"] < 0
)

loose_exact_ci_feasible = bool(
    k4_risk_ci_high <= LOOSE_BUDGET
)

strict_exact_ci_feasible = bool(
    k4_risk_ci_high <= STRICT_BUDGET
)

activity_ok = bool(
    activity_ratio
    >= V08_CFG["activity_floor_fraction"]
)

replication_confirmed = bool(
    len(k4_risk_by_seed)
    >= V08_CFG["minimum_complete_seeds"]
    and risk_reduction_confirmed
    and loose_exact_ci_feasible
    and activity_ok
)

route = (
    "ADVANCE_TO_FROZEN_OOS"
    if replication_confirmed
    else "BOUNDED_MULTI_SEED_RESULT_NO_RETUNING"
)

decision_df = pd.DataFrame([
    {
        "spec_id": SPEC_ID,
        "complete_seeds": len(k4_risk_by_seed),
        "selected_label": "K4_VERY_STRONG",
        "selected_kappa": SELECTED_KAPPA,
        "primary_checkpoint": 10,
        "sensitivity_checkpoint_evaluated": False,
        "primary_risk_difference": float(
            primary_risk["mean_difference"]
        ),
        "primary_risk_ci_low": float(
            primary_risk["ci_low"]
        ),
        "primary_risk_ci_high": float(
            primary_risk["ci_high"]
        ),
        "k4_primary_risk": k4_risk,
        "k4_primary_risk_ci_low": k4_risk_ci_low,
        "k4_primary_risk_ci_high": k4_risk_ci_high,
        "exact_loose_budget": LOOSE_BUDGET,
        "exact_loose_ci_feasible": loose_exact_ci_feasible,
        "exact_strict_budget": STRICT_BUDGET,
        "exact_strict_ci_feasible": strict_exact_ci_feasible,
        "activity_ratio": activity_ratio,
        "activity_floor_passed": activity_ok,
        "replication_confirmed": replication_confirmed,
        "next_route": route,
        "checkpoint_reselection_performed": False,
        "test_used": False,
    }
])

display(summary_df)
display(seed_effects_df)
display(comparison_df)
display(decision_df)

atomic_csv(
    episodes_df,
    V08_DIR / "primary_evaluation_episodes.csv",
)
atomic_parquet(
    transitions_df,
    V08_DIR / "primary_evaluation_transitions.parquet",
)
atomic_csv(
    summary_df,
    V08_DIR / "primary_evaluation_summary.csv",
)
atomic_csv(
    seed_effects_df,
    V08_DIR / "primary_seed_effects.csv",
)
atomic_csv(
    comparison_df,
    V08_DIR / "primary_hierarchical_comparison.csv",
)
atomic_csv(
    decision_df,
    V08_DIR / "primary_decision.csv",
)

decision_record = json.loads(
    decision_df.to_json(orient="records")
)[0]

temporary_path = V08_DIR / "primary_next_step.json.tmp"

with open(temporary_path, "w", encoding="utf-8") as file:
    json.dump(
        decision_record,
        file,
        ensure_ascii=False,
        indent=2,
    )

os.replace(
    temporary_path,
    V08_DIR / "primary_next_step.json",
)

print("=" * 76)
print("V0.8 PRIMARY ROUTE:", route)
print("replication confirmed:", replication_confirmed)
print(
    "primary Δrisk:",
    float(primary_risk["mean_difference"]),
)
print(
    "primary Δrisk CI:",
    [
        float(primary_risk["ci_low"]),
        float(primary_risk["ci_high"]),
    ],
)
print(
    "K4 risk / upper CI / loose budget:",
    k4_risk,
    k4_risk_ci_high,
    LOOSE_BUDGET,
)
print("activity ratio:", activity_ratio)
print("checkpoint 20 evaluated: False")
print("checkpoint reselection: False")
print("test used: False")
print("=" * 76)

,checkpoint_episode,label,risk,mean_pnl,mean_filled_notional,mean_abs_inventory,p95_abs_inventory,band_violation,action_saturation
0,10,K0_UNCONSTRAINED,3.617323,-10.538050,55514.811409,6.280498,18.318683,0.519665,0.340812
1,10,K4_VERY_STRONG,1.270385,-3.404397,42058.984994,2.164071,6.352836,0.224307,0.218029


,seed,checkpoint_episode,metric,mean_difference,k4_better_windows,windows
0,10801,10,normalized_discounted_cost,-2.966297,7,7
1,10802,10,normalized_discounted_cost,-4.253579,7,7
2,10803,10,normalized_discounted_cost,1.006805,0,7
3,10804,10,normalized_discounted_cost,-0.368462,5,7
4,10805,10,normalized_discounted_cost,-5.153156,7,7
5,10801,10,pnl_sum,-3.414988,3,7
6,10802,10,pnl_sum,22.685169,3,7
7,10803,10,pnl_sum,-3.311993,5,7
8,10804,10,pnl_sum,2.548286,3,7
9,10805,10,pnl_sum,17.161794,2,7


,checkpoint_episode,endpoint_role,metric,mean_difference,ci_low,ci_high,seeds,windows_per_seed
0,10,PRIMARY,normalized_discounted_cost,-2.346938,-4.381573,-0.224233,5,7
1,10,PRIMARY,pnl_sum,7.133653,-9.306681,25.410155,5,7


,spec_id,complete_seeds,selected_label,selected_kappa,primary_checkpoint,sensitivity_checkpoint_evaluated,primary_risk_difference,primary_risk_ci_low,primary_risk_ci_high,k4_primary_risk,...,exact_loose_budget,exact_loose_ci_feasible,exact_strict_budget,exact_strict_ci_feasible,activity_ratio,activity_floor_passed,replication_confirmed,next_route,checkpoint_reselection_performed,test_used
0,v08_frozen_k4_ep10_multiseed_20260925,5,K4_VERY_STRONG,0.013168,10,False,-2.346938,-4.381573,-0.224233,1.270385,...,1.214222,False,0.758888,False,0.757617,True,False,BOUNDED_MULTI_SEED_RESULT_NO_RETUNING,False,False


V0.8 PRIMARY ROUTE: BOUNDED_MULTI_SEED_RESULT_NO_RETUNING
replication confirmed: False
primary Δrisk: -2.3469377601028287
primary Δrisk CI: [-4.3815726817011855, -0.22423277343223633]
K4 risk / upper CI / loose budget: 1.2703849765597224 1.742751977338158 1.2142215137929977
activity ratio: 0.7576173624106906
checkpoint 20 evaluated: False
checkpoint reselection: False
test used: False
